# Benchmark 5: MCQ Standalone Validity and Shortcut Resistance

Implements Pillar 3 (Dual MCQ Evaluation) from the paper. Benchmark 5 evaluates
whether MCQ **options** - both correct and wrong - function as standalone,
contextually complete clinical statements independent of the multiple-choice
framing, and whether the correct option requires genuine visual grounding.

Unlike Benchmarks 1-4, **Benchmark 5 was never scored by human pathologists at
all** - the paper states explicitly (Section 3.5.4): "Since pathologist
recruitment for datasets outside PathVQA was not feasible at this scale,
Benchmark 5 scores for PathMMU and PatchVQA were obtained using an LLM-as-judge
protocol". So there is **no human baseline to validate against here**. Per
agreed scope, this notebook reports **judge-vs-judge agreement** (InternVL vs.
Qwen-VL) as the only reliability signal, and this is treated as an explicit
limitation, not a substitute for human validation.

**Critical methodological detail**: Benchmark 5a/5b score each option
**standalone** - "Standalone Completeness was scored using option text only,
without question-stem context" (Section 2.1, Sub-pillar 3a). This notebook's
prompts (`build_benchmark_5a_prompt` / `build_benchmark_5b_prompt`) never include
the question text, unlike every other benchmark notebook in this repo.

Scores **every resolvable item** across all three datasets (per user's explicit
choice - this is a large inference job, expect a long unattended run):
- **PathOPEN**: MCQ correct option (5a) + 4 wrong options (5b) per case, ~229
  resolvable cases.
- **PathMMU**: `val` + `test` + `test_tiny` splits across all 5 source categories
  (PubMed, SocialPath, EduContent, PathCLS, Atlas), correct option (5a) + all
  wrong options (5b, variable count per question).
- **PatchVQA**: same structure, 4 source categories.

**Known data gaps** (pre-existing, not introduced here): PathMMU's and
PatchVQA's `SocialPath` category is 100% unresolvable locally (images were never
downloaded - the README documents a separate manual acquisition step for this and
several `PathCLS` sub-datasets), and `PathCLS` is ~8.4% unresolvable. Unresolvable
items are skipped and counted, not silently dropped.

## Checkpointing

This is the **largest inference job in the whole pipeline** (thousands of MCQ
items, each with 1 correct + up to 5 wrong options, each option a separate judge
call, x2 judges x3 datasets). Every individual judge call is checkpointed
immediately to `checkpoints/{model_key}_benchmark5.jsonl`, keyed by
`item_id = "{dataset}::{category}::{split}::{No}::{option_slot}"` (`option_slot`
is `"correct"` or `"wrong_N"`). Re-running any scoring cell after an
interruption skips everything already checkpointed and only scores what's
missing - given the scale here, this is the single most important notebook to
run with checkpointing, since an unattended multi-hour run failing near the end
without it would be very costly to redo.


In [ ]:
import os
# Force the system to only see GPU #0. (Change to 0, 2, 3 etc. as needed)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
import glob
import json as _json
import os

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from checkpoint import JudgeCheckpoint
from judge_models import JudgeModel
from prompts.benchmarks import (
    BENCHMARK_5A,
    BENCHMARK_5B,
    build_benchmark_5a_prompt,
    build_benchmark_5b_prompt,
)


In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SUBSETS_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "subsets_processing_output", "data"
)
PROCESSED_DATA_JSON = os.path.join(REPO_ROOT, "pathopen_data", "processed", "data.json")

PATHMMU_ROOT = "/data/mn27889/pathology-datasets/PathMMU"
PATHMMU_DATA_JSON = os.path.join(PATHMMU_ROOT, "data.json")
PATHMMU_IMAGES_DIR = os.path.join(PATHMMU_ROOT, "images")

PATCHVQA_JSON = "/data/mn27889/pathology-datasets/PathBench/data/PatchVQA.json"
# PatchVQA's images are sourced from PathMMU's image pool (documented in PathBench's
# own README: "Access images can from PathMMU"), not a separate directory.
PATCHVQA_IMAGES_DIR = PATHMMU_IMAGES_DIR

OUTPUT_DIR = os.path.join(os.getcwd(), "benchmark5_output")
CHECKPOINT_DIR = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

REPO_ROOT, SUBSETS_DIR, PATHMMU_DATA_JSON, PATCHVQA_JSON, CHECKPOINT_DIR


## Load judge models

In [ ]:
MODELS_TO_RUN = ["internvl", "qwenvl"]

judges = {key: JudgeModel(key) for key in MODELS_TO_RUN}
judges


## PathOPEN: MCQ options, standalone

Reuses the same `pathopen_vqa_part{1-5}.csv` source subset and `processed/data.json`
image resolver as `judge_runner_pathopen.ipynb`, but scores MCQ options standalone
(no question text in the prompt) rather than as open-ended Q&A.

In [ ]:
with open(PROCESSED_DATA_JSON) as f:
    _processed_cases = _json.load(f)
_processed_root = os.path.dirname(PROCESSED_DATA_JSON)

PATHOPEN_IMAGE_INDEX = {}
for _case in _processed_cases:
    for _img in _case["Images"]:
        original = _img.get("Original", "")
        if not original:
            continue
        image_id = original.rsplit("_orig.", 1)[0]
        rel_dir = _img["Directory"].split("processed/")[-1]
        abs_dir = os.path.normpath(os.path.join(_processed_root, rel_dir))
        PATHOPEN_IMAGE_INDEX[image_id] = {"dir": abs_dir, "original": original}


def load_pathopen_image(image_id: str) -> Image.Image:
    entry = PATHOPEN_IMAGE_INDEX[image_id]
    return Image.open(os.path.join(entry["dir"], entry["original"])).convert("RGB")


core_parts = sorted(glob.glob(os.path.join(SUBSETS_DIR, "pathopen_vqa_part*.csv")))
pathopen_df = pd.concat([pd.read_csv(p) for p in core_parts], ignore_index=True)
pathopen_df = pathopen_df[pathopen_df["Image_ID"].isin(PATHOPEN_IMAGE_INDEX)]
len(pathopen_df)


In [ ]:
def iter_pathopen_mcq_subtasks(row: pd.Series):
    """Yields (option_slot, prompt, criteria, option_text) for one PathOPEN row's
    correct option (5a) and 4 wrong options (5b)."""
    yield (
        "correct",
        build_benchmark_5a_prompt(row["MCQ_OE_Correct_Answer"]),
        list(BENCHMARK_5A["criteria"].keys()),
        row["MCQ_OE_Correct_Answer"],
    )
    for i in (1, 2, 3, 4):
        wrong_col = f"MCQ_OE_Wrong_Answer_{i}"
        yield (
            f"wrong_{i}",
            build_benchmark_5b_prompt(row[wrong_col]),
            list(BENCHMARK_5B["criteria"].keys()),
            row[wrong_col],
        )


def run_pathopen_benchmark5(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_benchmark5.jsonl"))
    n_scored = n_skipped = n_failed = 0

    for _, row in tqdm(pathopen_df.iterrows(), total=len(pathopen_df), desc=f"{model_key} PathOPEN Benchmark5"):
        image = None
        for option_slot, prompt, criteria, option_text in iter_pathopen_mcq_subtasks(row):
            item_id = f"PathOPEN::None::None::{row['Image_ID']}::{option_slot}"
            if checkpoint.is_done(item_id):
                n_skipped += 1
                continue
            try:
                if image is None:
                    image = load_pathopen_image(row["Image_ID"])
                scores, raw = judge.score(image, prompt, criteria)
            except Exception as e:
                n_failed += 1
                print(f"[checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
                continue
            checkpoint.append({
                "item_id": item_id,
                "dataset": "PathOPEN",
                "source_category": None,
                "split": None,
                "row_no": row["Image_ID"],
                "option_slot": option_slot,
                "option_text": option_text,
                "scores": scores,
                "raw_response": raw,
            })
            n_scored += 1

    print(f"{model_key} PathOPEN Benchmark5: scored {n_scored} new, {n_skipped} already done, {n_failed} failed this run")
    return checkpoint


pathopen_b5_checkpoints = {model_key: run_pathopen_benchmark5(judges[model_key], model_key) for model_key in MODELS_TO_RUN}


## PathMMU and PatchVQA: MCQ options, standalone

Both datasets share the same JSON shape: `{category: {split: [ {img, question,
options, answer}, ... ]}}`, with `options` as letter-prefixed strings (e.g. `"A)
..."`) and `answer` matching one option's full text exactly. Wrong options are
simply every option that isn't the answer - a variable count (4-6 for PathMMU/
PatchVQA vs. PathOPEN's fixed 5).

In [ ]:
def load_pathmmu_style_dataset(json_path: str) -> list:
    """Flattens {category: {split: [items]}} into a flat list of dict records,
    each tagged with its source category and split."""
    with open(json_path) as f:
        data = _json.load(f)
    records = []
    for category, splits in data.items():
        for split, items in splits.items():
            for item in items:
                records.append({**item, "_category": category, "_split": split})
    return records


def resolve_pathmmu_style_image(images_dir: str, img_filename: str):
    path = os.path.join(images_dir, img_filename)
    if not os.path.exists(path):
        return None
    return Image.open(path).convert("RGB")


pathmmu_items = load_pathmmu_style_dataset(PATHMMU_DATA_JSON)
patchvqa_items = load_pathmmu_style_dataset(PATCHVQA_JSON)
len(pathmmu_items), len(patchvqa_items)


In [ ]:
def iter_pathmmu_style_subtasks(item: dict):
    options = item["options"]
    answer = item["answer"]
    wrong_options = [opt for opt in options if opt != answer]

    yield ("correct", build_benchmark_5a_prompt(answer), list(BENCHMARK_5A["criteria"].keys()), answer)
    for i, wrong_option in enumerate(wrong_options, start=1):
        yield (
            f"wrong_{i}",
            build_benchmark_5b_prompt(wrong_option),
            list(BENCHMARK_5B["criteria"].keys()),
            wrong_option,
        )


def run_pathmmu_style_benchmark5(judge: JudgeModel, items: list, images_dir: str, dataset_name: str, model_key: str, checkpoint: JudgeCheckpoint) -> None:
    n_scored = n_skipped = n_failed = n_unresolvable = 0

    for item in tqdm(items, desc=f"{model_key} {dataset_name} Benchmark5"):
        item_no = item.get("No", item["img"])
        base_id = f"{dataset_name}::{item['_category']}::{item['_split']}::{item_no}"
        # Skip the whole item cheaply if every subtask is already checkpointed,
        # without touching disk/image loading at all.
        subtasks = list(iter_pathmmu_style_subtasks(item))
        if all(checkpoint.is_done(f"{base_id}::{slot}") for slot, *_ in subtasks):
            n_skipped += len(subtasks)
            continue

        image = resolve_pathmmu_style_image(images_dir, item["img"])
        if image is None:
            n_unresolvable += 1
            continue

        for option_slot, prompt, criteria, option_text in subtasks:
            item_id = f"{base_id}::{option_slot}"
            if checkpoint.is_done(item_id):
                n_skipped += 1
                continue
            try:
                scores, raw = judge.score(image, prompt, criteria)
            except Exception as e:
                n_failed += 1
                print(f"[checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
                continue
            checkpoint.append({
                "item_id": item_id,
                "dataset": dataset_name,
                "source_category": item["_category"],
                "split": item["_split"],
                "row_no": item_no,
                "option_slot": option_slot,
                "option_text": option_text,
                "scores": scores,
                "raw_response": raw,
            })
            n_scored += 1

    print(f"{model_key} {dataset_name} Benchmark5: scored {n_scored} new, {n_skipped} already done, "
          f"{n_unresolvable} items unresolvable (image not found), {n_failed} failed this run")


In [ ]:
for model_key in MODELS_TO_RUN:
    checkpoint = pathopen_b5_checkpoints[model_key]  # same checkpoint file covers all 3 datasets
    run_pathmmu_style_benchmark5(judges[model_key], pathmmu_items, PATHMMU_IMAGES_DIR, "PathMMU", model_key, checkpoint)
    run_pathmmu_style_benchmark5(judges[model_key], patchvqa_items, PATCHVQA_IMAGES_DIR, "PatchVQA", model_key, checkpoint)


## Post-process: assemble the checkpoint into per-dataset CSVs

Pure re-read of the checkpoint **file** (via a fresh `JudgeCheckpoint(path)`, not
the in-memory `pathopen_b5_checkpoints` object from the scoring cells above) -
safe to re-run any time, independent of the scoring cells above, even in a fresh
kernel (no judge models or GPU needed, just `os`/`pandas`/`json` and the
`CHECKPOINT_DIR`/`OUTPUT_DIR` paths above).

In [ ]:
def assemble_benchmark5_csv(model_key: str, dataset_name: str) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_benchmark5.jsonl directly from disk (one
    shared checkpoint file covers all 3 datasets) - does NOT depend on the
    `pathopen_b5_checkpoints` dict from the scoring cells, so this is safe to run
    standalone in a fresh kernel."""
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_benchmark5.jsonl"))
    rows_by_key = {}
    for record in checkpoint.load_all():
        if record["dataset"] != dataset_name:
            continue
        key = (record["source_category"], record["split"], record["row_no"])
        row = rows_by_key.setdefault(key, {
            "dataset": dataset_name,
            "source_category": record["source_category"],
            "split": record["split"],
            "item_id": record["row_no"],
        })
        slot = record["option_slot"]
        if slot == "correct":
            row["correct_option_text"] = record["option_text"]
            row["correct_Standalone_Completeness"] = record["scores"].get("Standalone Completeness")
            row["correct_Visual_Grounding"] = record["scores"].get("Visual Grounding")
        else:
            row[f"{slot}_option_text"] = record["option_text"]
            row[f"{slot}_Distractor_Standalone_Plausibility"] = record["scores"].get("Distractor Standalone Plausibility")
            row[f"{slot}_Distractor_Visual_Grounding_Error"] = record["scores"].get("Distractor Visual Grounding Error")
    return pd.DataFrame.from_records(list(rows_by_key.values()))


for model_key in MODELS_TO_RUN:
    for dataset_name in ["PathOPEN", "PathMMU", "PatchVQA"]:
        out_df = assemble_benchmark5_csv(model_key, dataset_name)
        out_path = os.path.join(OUTPUT_DIR, f"{model_key}_{dataset_name.lower()}_benchmark5.csv")
        out_df.to_csv(out_path, index=False)
        print(model_key, dataset_name, "->", out_path, out_df.shape)


## Judge-vs-judge agreement (the only reliability signal available for Benchmark 5)

Since there is no human ground truth for Benchmark 5, this computes weighted
Cohen's kappa between InternVL and Qwen-VL's scores on the **same items**, per
dataset and per criterion, as a lower bar of "do two independent judges even
agree with each other" - not proof of correctness.

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

VALID_SCORES = {-1, 0, 1, 2}


def judge_vs_judge_kappa(df_a: pd.DataFrame, df_b: pd.DataFrame, join_cols: list, score_col: str) -> dict:
    merged = df_a.merge(df_b, on=join_cols, suffixes=("_a", "_b"))
    a = pd.to_numeric(merged[f"{score_col}_a"], errors="coerce")
    b = pd.to_numeric(merged[f"{score_col}_b"], errors="coerce")
    valid = a.isin(VALID_SCORES) & b.isin(VALID_SCORES)
    n = int(valid.sum())
    if n < 2:
        return {"n": n, "weighted_kappa": np.nan}
    kappa = cohen_kappa_score(a[valid].astype(int), b[valid].astype(int), weights="quadratic")
    return {"n": n, "weighted_kappa": kappa}


jvj_rows = []
for dataset_name, join_cols in [
    ("PathOPEN", ["item_id"]),
    ("PathMMU", ["item_id", "source_category", "split"]),
    ("PatchVQA", ["item_id", "source_category", "split"]),
]:
    fname = f"{dataset_name.lower()}_benchmark5.csv"
    df_internvl = pd.read_csv(os.path.join(OUTPUT_DIR, f"internvl_{fname}"))
    df_qwenvl = pd.read_csv(os.path.join(OUTPUT_DIR, f"qwenvl_{fname}"))
    for score_col in ["correct_Standalone_Completeness", "correct_Visual_Grounding"]:
        result = judge_vs_judge_kappa(df_internvl, df_qwenvl, join_cols, score_col)
        jvj_rows.append({"dataset": dataset_name, "criterion": score_col, **result})

judge_vs_judge_df = pd.DataFrame(jvj_rows)
judge_vs_judge_df


In [ ]:
judge_vs_judge_df.to_csv(os.path.join(OUTPUT_DIR, "judge_vs_judge_kappa.csv"), index=False)


## Limitations (explicit, per agreed scope)

- **No human baseline for Benchmark 5** - the reported judge-vs-judge kappa is a
  weak reliability signal (two models could share the same blind spots) and
  should not be reported as validation in the same sense as the Benchmark 1-4
  kappas in `judge_pathologist_agreement.ipynb`.
- **SocialPath category unresolvable** for both PathMMU and PatchVQA (documented
  data-acquisition gap, images never downloaded locally) - excluded entirely,
  not scored.
- **PathCLS category ~8.4% unresolvable** - individual items skipped and counted,
  not silently dropped.

## Raw checkpoint files

`checkpoints/{model_key}_benchmark5.jsonl` retains every judge call across all
three datasets (option text, scores, and full raw model response) and is never
overwritten - it is the single source of truth this notebook's CSVs and kappa
table are derived from, and can be re-processed at any time without re-running
any inference.
